In [1]:
import torch
import pyro
import pyro.distributions as dist
from pyro.distributions.transforms import AffineTransform
from pyro.distributions import TransformedDistribution, Normal
#from pyro.infer.reparam import LocScaleReparam
from pyro.infer.reparam.reparam import Reparam
from chirho.observational.handlers.condition import condition
from chirho.interventional.handlers import do
from chirho.counterfactual.handlers import MultiWorldCounterfactual

In [ ]:
class NormalReparam(Reparam):

    def __init__(self,):
        super().__init__()


    def apply(self, msg):
        name = msg["name"]
        fn = msg["fn"]
        value = msg["value"]
        is_observed = msg["is_observed"]

        msg

        loc = fn.loc
        scale = fn.scale

        shape = value.shape if value is not None else fn.batch_shape + fn.event_shape

        event_dim = fn.event_dim

        with pyro.poutine.mask(mask=False):
            base_noise = pyro.sample(
                f"{name}_base_noise",
                dist.Normal(torch.zeros(shape), torch.ones(shape)),
            ).to(loc.device)

        if is_observed:
            new_value = value

        else:
            transform = AffineTransform(loc, scale, event_dim=event_dim)
            new_value = transform(base_noise)
        
        
        return {"fn": fn, "value": new_value, "is_observed": is_observed}


In [3]:
with MultiWorldCounterfactual() as mc:
    with do(actions={"x": torch.tensor(2.0)}):
            with pyro.poutine.reparam(config={"x": NormalReparam()}):
                with pyro.plate("data_plate", 10):
                #with condition(data={"x": torch.tensor(1.0)}):
                    with pyro.poutine.trace() as tr:
                        x = pyro.sample("x", dist.Normal(5, 2))


NameError: name 'centered' is not defined

In [ ]:


    #with pyro.poutine.reparam(config={"x": NormalReparam()}):
with MultiWorldCounterfactual() as mc:
    with do(actions={"x": torch.tensor(2.0)}):
            with pyro.poutine.reparam(config={"x": NormalReparam()}):
                with pyro.plate("data_plate", 10):
                #with condition(data={"x": torch.tensor(1.0)}):
                    with pyro.poutine.trace() as tr:
                        x = pyro.sample("x", dist.Normal(5, 2))

print(tr.trace.nodes.keys())
tr.trace.nodes.keys()

x_traced = tr.trace.nodes["x"]['value']
base_noise_traced = tr.trace.nodes["x_base_noise"]["value"]

print(x_traced.shape)
print(base_noise_traced.shape)


/home/rafal/.local/lib/python3.10/site-packages/pyro/poutine/reparam_messenger.py:135: RuntimeWarning: At pyro.sample('x',...), LocScaleReparam does not commute with initialization; falling back to default initialization.
  warnings.warn(


odict_keys(['base_noise', 'x'])

In [ ]:
reparam_model = poutine.reparam(model, {"v": DiscreteCosineReparam(),
                                        "r": StableReparam()})

with poutine.reparam(config={"drift": LocScaleReparam()}):
            drift = pyro.sample("drift", dist.Normal(zero_data, drift_scale).to_event(1))


In [ ]:
class KernelSoftConditionReparam(pyro.infer.reparam.reparam.Reparam):
    """
    Reparametrizer that allows approximate soft conditioning on a :func:`pyro.deterministic`
    site using a kernel function that compares the observed and computed values,
    as in approximate Bayesian computation methods from classical statistics.

    This may be useful for estimating counterfactuals in Pyro programs
    corresponding to structural causal models with exogenous noise variables.

    The kernel function should return a score corresponding to the
    log-probability of the observed value given the computed value,
    which is then added to the model's unnormalized log-joint probability
    using :func:`pyro.factor`  :

        :math:`\\log p(v' | v) \\approx K(v, v')`

    The score tensor returned by the kernel function must have shape equal
    or broadcastable to the ``batch_shape`` of the site.

    .. note::
        Kernel functions must be positive-definite and symmetric.
        For example, :class:`~RBFKernel` returns a Normal log-probability
        of the distance between the observed and computed values.
    """

    def __init__(self, kernel: Kernel[torch.Tensor]):
        self.kernel = kernel
        super().__init__()

    def apply(self, msg: pyro.infer.reparam.reparam.ReparamMessage) -> pyro.infer.reparam.reparam.ReparamResult:
        assert isinstance(msg["fn"], TorchDistributionMixin)
        assert msg["value"] is not None

        name = msg["name"]
        event_dim = msg["fn"].event_dim
        observed_value = msg["value"]
        computed_value = msg["fn"].base_dist.v

        if observed_value is not computed_value:  # fast path for trivial case
            approx_log_prob = self.kernel(computed_value, observed_value)
            pyro.factor(f"{name}_approx_log_prob", approx_log_prob)

        new_fn = pyro.distributions.Delta(observed_value, event_dim=event_dim).mask(False)
        return {"fn": new_fn, "value": observed_value, "is_observed": True}


In [ ]:
class LocScaleReparam(Reparam):


    def __init__(self, centered=None, shape_params=None):
        assert centered is None or isinstance(centered, (float, torch.Tensor))
        if shape_params is not None:
            assert isinstance(shape_params, (tuple, list))
            assert all(isinstance(name, str) for name in shape_params)
        if is_validation_enabled():
            if isinstance(centered, float):
                assert 0 <= centered and centered <= 1
            elif isinstance(centered, torch.Tensor):
                assert (0 <= centered).all()
                assert (centered <= 1).all()
            else:
                assert centered is None
        self.centered = centered
        self.shape_params = shape_params


    def apply(self, msg):
        name = msg["name"]
        fn = msg["fn"]
        value = msg["value"]
        is_observed = msg["is_observed"]

        centered = self.centered
        if is_identically_one(centered):
            return msg
        event_shape = fn.event_shape
        fn, event_dim = self._unwrap(fn)

        # Apply a partial decentering transform.
        if self.shape_params is None:
            self.shape_params = tuple(
                k for k in fn.arg_constraints if k not in ("loc", "scale")
            )
        params = {key: getattr(fn, key) for key in self.shape_params}
        if centered is None:
            centered = pyro.param(
                "{}_centered".format(name),
                lambda: fn.loc.new_full(event_shape, 0.5),
                constraint=constraints.unit_interval,
            )
        params["loc"] = fn.loc * centered
        params["scale"] = fn.scale**centered
        decentered_fn = type(fn)(**params)

        # Differentiably invert transform.
        decentered_value = None
        if value is not None:
            delta = (value - fn.loc) * fn.scale.pow(centered - 1)
            decentered_value = delta + centered * fn.loc

        # Draw decentered noise.
        decentered_value = pyro.sample(
            f"{name}_decentered",
            self._wrap(decentered_fn, event_dim),
            obs=decentered_value,
            infer={"is_observed": is_observed},
        )

        # Differentiably transform.
        if value is None:
            delta = decentered_value - centered * fn.loc
            value = fn.loc + fn.scale.pow(1 - centered) * delta

        # Simulate a pyro.deterministic() site.
        new_fn = dist.Delta(value, event_dim=event_dim).mask(False)
        return {"fn": new_fn, "value": value, "is_observed": True}

In [5]:
with pyro.poutine.trace() as tr:
    z = pyro.sample(
        "z",
        TransformedDistribution(
            Normal(0, 1),  # <-- positional base distribution
            [AffineTransform(loc=2.0, scale=3.0)]  # <-- list of transforms
        )
    )

print(tr.trace.nodes.keys())

odict_keys(['z'])
